Tutorial 2: Practicalities
==========================

In the last tutorial, we introduced foundational statistical concepts essential for model-fitting, such as parameter
spaces, likelihoods, priors, and non-linear searches. Understanding these statistical concepts is crucial for
performing model fits effectively.

However, achieving successful model-fitting also requires practical skills, including how to manage outputs,
review results, interpret model quality, and ensure run-times are efficient enough for your scientific needs.

This tutorial will focus on these practical aspects of model-fitting, including:

- How to save results to your hard disk.
- How to navigate the output folder and examine model-fit results to assess quality.
- How to estimate the run-time of a model-fit before initiating it, and change settings to make it faster or run the analysis in parallel.

__Contents__

- **PyAutoFit:** Modeling uses the probabilistic programming language.
- **Initial Setup:** Lets first load the `Imaging` dataset we'll fit a model with using a non-linear search.
- **Dataset Auto-Simulation:** Simulate the dataset via its simulator script if it is not on your hard-disk.
- **Mask:** Define the 2D mask applied to the dataset for the model-fit.
- **Model:** Compose the lens model fitted to the data.
- **Search:** Configure the non-linear search used to fit the model.
- **Search Settings:** Nautilus samples parameter space by placing "live points" representing different galaxy models.
- **Iterations Per Update:** Every N iterations, the non-linear search outputs the current results to the folder.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **VRAM Use:** When running AutoLens with JAX on a GPU, the analysis must fit within the GPU’s available VRAM.
- **Run Times:** Profiling the expected run time of the model-fit.
- **Model-Fit:** Begin the model-fit by passing the model and analysis to the search.
- **Result Info:** A concise readable summary of the results is given by printing its `info` attribute.
- **Output Folder:** Now checkout the `output` folder.
- **Unique Identifier:** In the output folder, you will note that results are in a folder which is a collection of random.
- **Output Folder Contents:** Now this is running you should checkout the `output` folder.
- **Result:** Overview of the results of the model-fit.
- **Other Practicalities:** The following are examples of other practicalities which I will document fully in this example.
- **Wrap Up:** Summary of the script and next steps.

__Search Settings__

Nautilus samples parameter space by placing "live points" representing different galaxy models. Each point has an
associated `log_likelihood` that reflects how well it fits the data. By mapping where high-likelihood solutions are
located, it can focus on searching those regions.

The main setting to balance is the **number of live points**. More live points allow Nautilus to map parameter space
more thoroughly, increasing accuracy but also runtime. Fewer live points reduce run-time but may make the search less
reliable, possibly getting stuck in local maxima.

The ideal number of live points depends on model complexity. More parameters generally require more live points, but
the default of 200 is sufficient for most lens models. Lower values can still yield reliable results, particularly
for simpler models. For this example (6 parameters), we reduce the live points to 100 to speed up runtime without
compromising accuracy.

Tuning non-linear search settings (e.g., the number of live points) to match model complexity is essential. We aim
for enough live points to ensure accurate results (i.e., finding a global maximum) but not so many that runtime
is excessive.

In practice, the optimal number of live points is often found through trial and error, guided by summary statistics
on how well the search is performing, which we’ll cover below. For the simple model fitted here, with a linear light
profile, 100 live points is sufficient to achieve reliable results.

__Iterations Per Update__

Every N iterations, the non-linear search outputs the current results to the folder `output`,
which includes producing visualization.

Depending on how long it takes for the model to be fitted to the data (see discussion about run times below),
this can take up a large fraction of the run-time of the non-linear search.

For this fit, the fit is very fast, thus we set a high value of `iterations_per_quick_update=2500` to ensure these updates
do not slow down the overall speed of the model-fit.

**If the iteration per update is too low, the model-fit may be significantly slowed down by the time it takes to
output results and visualization frequently to hard-disk. If your fit is consistent displaying a log saying that it
is outputting results, try increasing this value to ensure the model-fit runs efficiently.**

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__PyAutoFit__

Modeling uses the probabilistic programming language
[PyAutoFit](https://github.com/PyAutoLabs/PyAutoFit), an open-source project that allows complex model
fitting techniques to be straightforwardly integrated into scientific modeling software.

The majority of tools that make model-fitting practical are provided by PyAutoFit, for example it handles
all output of the non-linear search to hard-disk, the visualization of results and the estimation of run-times.

In [ ]:
import autofit as af

__Initial Setup__

Lets first load the `Imaging` dataset we'll fit a model with using a non-linear search.

This is the same dataset we fitted in the previous tutorial, and we'll repeat the same fit, as we simply want
to illustrate the practicalities of model-fitting in this tutorial.

In [ ]:
dataset_name = "simple__no_lens_light__mass_sis"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/simulator/no_lens_light__mass_sis.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

The non-linear fit also needs a `Mask2D`, lets use a 3.0" circle.

In [ ]:
mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=dataset)

__Model__

We compose the model using the same API as the previous tutorial.

This model is the same as the previous tutorial, an `IsothermalSph` spherical mass profile representing the lens
galaxy and a `ExponentialCoreSph` light profile representing the source galaxy.

In [ ]:
# Lens:

mass = af.Model(al.mp.IsothermalSph)

lens = af.Model(al.Galaxy, redshift=0.5, mass=mass)

# Source:

bulge = af.Model(al.lp_linear.ExponentialCoreSph)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

print(model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free,
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed
parameter.

Nothing in this tutorial changes the model — every setting we discuss below belongs to the search.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

We again use the `Nautilus` non-linear search, with settings chosen following the Search Settings discussion above.

In [ ]:
search = af.Nautilus(
    path_prefix=Path("howtolens") / "chapter_2",
    name="tutorial_2_practicalities",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU batching and VRAM use explained in VRAM section below.
    iterations_per_quick_update=2500,  # Outputs Notebook visualization of max likelihood model every N iterations
)

__Analysis__

We again create the `AnalysisImaging` object which contains the `log_likelihood_function` that the non-linear search
calls to fit the model to the data.

In [ ]:
analysis = al.AnalysisImaging(dataset=dataset)

__VRAM Use__

When running AutoLens with JAX on a GPU, the analysis must fit within the GPU’s
available VRAM. If insufficient VRAM is available, the analysis will fail with an
out-of-memory error, typically during JIT compilation or the first likelihood call.

Two factors dictate the VRAM usage of an analysis:

- The number of arrays and other data structures JAX must store in VRAM to fit the model
  to the data in the likelihood function. This is dictated by the model complexity and dataset size.

- The `batch_size` sets how many likelihood evaluations are performed simultaneously.
  Increasing the batch size increases VRAM usage but can reduce overall run time,
  while decreasing it lowers VRAM usage at the cost of slower execution.

Before running an analysis, users should check that the estimated VRAM usage for the
chosen batch size is comfortably below their GPU’s total VRAM.

The method below prints the VRAM usage estimate for the analysis and model with the specified batch size,
it takes about 20-30 seconds to run so you may want to comment it out once you are familiar with your GPU's VRAM limits.

For the simple model and low resolution dataset fitted in this example VRAM use is relatively low (~0.027GB). For other
models (e.g. pixelized sources) and higher resolution datasets it can be much higher (> 1GB going beyond 10GB).

In [ ]:
analysis.print_vram_use(model=model, batch_size=search.batch_size)

__Run Times__

Lens modeling can be a computationally expensive process. When fitting complex models to high resolution datasets 
run times can be of order hours, days, weeks or even months.

Run times are dictated by two factors:

 - **The log likelihood evaluation time:** the time it takes for a single `instance` of the model to be fitted to 
   the dataset such that a log likelihood is returned.

 - **The number of iterations (e.g. log likelihood evaluations) performed by the non-linear search:** more complex lens
   models require more iterations to converge to a solution (and as discussed above, settings like the number of live
   points also control this).

For this analysis, the log likelihood evaluation time is < 0.01 seconds on CPU, < 0.001 seconds on GPU, which is 
extremely fast for lens modeling. 

The more advanced fitting techniques discussed at the end of chapter 1 (e.g. shapelets, multi Gaussian expansions, 
pixelizations) have longer log likelihood evaluation times. However, on GPU, they can be so fast they may not produce
significantly longer overall run-time feasible.

To estimate the expected overall run time of the model-fit we multiply the log likelihood evaluation time by an 
estimate of the number of iterations the non-linear search will perform, which is around 50000 to 100000 for this model.

Tests thus far reveal CPU run times are around 30 minutes, and GPU run times around 10 minutes.

__Model-Fit__

To begin the model-fit, we pass the model and analysis objects to the search, which performs a non-linear search to 
identify models that best fit the data.

Running model fits via non-linear search can take significant time. While the fit in this tutorial should 
complete in a few minutes, more complex models may require longer run times. In Jupyter notebooks, this can be 
limiting, as the notebook cell will only complete once the fit finishes, preventing you from advancing through the 
tutorial or running additional code.

To work around this, we recommend running tutorial scripts as standalone Python scripts, found
in the `scripts/` folder of the HowToLens repository. These scripts mirror the notebook tutorials but run independently of
Jupyter notebooks. For example, you can start a script with:

`python3 scripts/chapter_2_lens_modeling/tutorial_2_practicalities.py`

Using scripts allows results to be saved to the hard drive in the `output` folder, enabling you to inspect results 
immediately once the script completes. When rerun, the script loads results directly from disk, so any Jupyter 
notebook cells will quickly load and display the complete model-fit results if they’re already saved.

This approach not only avoids the slowdowns associated with notebook cells during lengthy runs but is also essential 
for using super-computers for fitting tasks, as they require separate Python scripts.

For tasks like loading results, inspecting data, plotting, and interpreting results, Jupyter notebooks remain ideal.

In [ ]:
print(
    "The non-linear search has begun running - checkout the output/howtolens/chapter_2/tutorial_2_practicalities"
    " folder for live output of the results, images and lens model."
    " This Jupyter notebook cell with progress once search has completed - this could take some time!"
)

result = search.fit(model=model, analysis=analysis)

print("Search has finished run - you may now continue the notebook.")

__Result Info__

A concise readable summary of the results is given by printing its `info` attribute.

[Above, we discussed that the `info_whitespace_length` parameter in the config files could b changed to make 
the `model.info` attribute display optimally on your computer. This attribute also controls the whitespace of the
`result.info` attribute.]

In [ ]:
print(result.info)

__Output Folder__

Now checkout the `output` folder.

This is where the results of the search are written to hard-disk (in the `tutorial_2_practicalities` folder). 

Once completed images, results and information about the fit appear in this folder, meaning that you don't need 
to keep running Python code to see the result.

__Unique Identifier__

In the output folder, you will note that results are in a folder which is a collection of random characters. This acts 
as a `unique_identifier` of the model-fit, where this identifier is generated based on the model, search and dataset 
that are used in the fit.
 
An identical combination of model, search and dataset generates the same identifier, meaning that rerunning the
script will use the existing results to resume the model-fit. In contrast, if you change the model, search or dataset,
a new unique identifier will be generated, ensuring that the model-fit results are output into a separate folder. 

We additionally want the unique identifier to be specific to the dataset fitted, so that if we fit different datasets
with the same model and search results are output to a different folder. We achieved this for the fit above by passing 
the `dataset_name` to the search's `unique_tag`.

__Output Folder Contents__

Now this is running you should checkout the `output` folder. This is where the results of the
search are written to hard-disk (in the `tutorial_2_practicalities` folder), where all outputs are human readable
(e.g. as .json, .csv or text files).

As the fit progresses, results are written to the `output` folder on the fly using the highest likelihood model found
by the non-linear search so far. This means you can inspect the results of the model-fit as it runs, without having to
wait for the non-linear search to terminate.
 
The `output` folder includes:

 - `model.info`: Summarizes the model, its parameters and their priors discussed in the next tutorial.
 
 - `model.results`: Summarizes the highest likelihood model inferred so far including errors.
 
 - `images`: Visualization of the highest likelihood model-fit to the dataset, (e.g. a fit subplot showing the 
 galaxies, model data and residuals).
 
 - `files`: A folder containing .fits files of the dataset, the model as a human-readable .json file, 
 a `.csv` table of every non-linear search sample and other files containing information about the model-fit.
 
 - `search.summary`: A file providing summary statistics on the performance of the non-linear search.
 
 - `search_internal`: Internal files of the non-linear search (in this case Nautilus) used for resuming the fit and
  visualizing the search.

__Result__

The `search.fit` method produces a `result` object, packed with useful information about the model fit that we’ll 
explore in detail in a later tutorial.

One component of the `result` object we’ll use now is the `FitImaging` object, which corresponds to the set of model 
parameters that yielded the maximum log-likelihood solution. Plotting this object lets us visually inspect how well 
the model fits the data.

In this example, the fit to the data is excellent, with residuals near zero, as expected since the same model was 
used both to simulate and fit the data.

In [ ]:
aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

The Probability Density Functions (PDF's) of the results can be plotted using an in-built visualization 
library, which is wrapped via the `corner_anesthetic` function.

The PDF shows the 1D and 2D probabilities estimated for every parameter after the model-fit. The two dimensional 
figures can show the degeneracies between different parameters, for example how increasing the intensity $I$ of the
source galaxy and decreasing its effective radius $R_{Eff}$ lead to similar likelihoods and probabilities.

This PDF will be discussed more in the next tutorial.

The plot is labeled with short hand parameter names (e.g. `sersic_index` is mapped to the short hand 
parameter `n`). These mappings ate specified in the `config/notation.yaml` file and can be customized by users.

The superscripts of labels correspond to the name each component was given in the model (e.g. for the `Isothermal`
mass its name `mass` defined when making the `Model` above is used).

In [ ]:
aplt.corner_anesthetic(samples=result.samples)

__Other Practicalities__

The following are examples of other practicalities which I will document fully in this example script in the future,
but so far have no found the time:

- `config`: The files in the `config` folder which control many aspects of how **PyAutoLens** runs,
 including visualization, the non-linear search settings.

- `config/priors`: Folder containing the default priors on all model components.

- `results`: How to load the results of a model-fit from the output folder to a Python script or Jupyter notebook.

- `output.yaml`: What files are output to control file size.

__Wrap Up__

This tutorial has illustrated how to handle a number of practicalities that are key to performing model-fitting
effectively. These include:

- How to save results to your hard disk.

- How to navigate the output folder and examine model-fit results to assess quality.

- How to estimate the run-time of a model-fit before initiating it, and change settings to make it faster or run the
  analysis in parallel.